In [1]:
import torch
import numpy as np


In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

get_device = get_device()
get_device

device(type='cuda')

In [3]:
import pandas as pd

try: # KAGGLE
    df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
except: 
    try: # COLAB
        df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
    except: # LOCAL
        df = pd.read_csv('../data/train.csv')
        test_df = pd.read_csv('../data/test.csv')
        
df.shape, test_df.shape

((2000, 8), (500, 7))

---
# EDA

In [4]:
df.isna().sum(), test_df.isna().sum()


(id        0
 prompt    0
 A         0
 B         0
 C         0
 D         0
 E         0
 answer    0
 dtype: int64,
 id        0
 prompt    0
 A         0
 B         0
 C         0
 D         0
 E         0
 dtype: int64)

In [5]:
# checking if test_df and df has similar rows
similar_rows = test_df.merge(df, on=['prompt'], how='inner')

similar_rows

,id_x,prompt,A_x,B_x,C_x,D_x,E_x,id_y,A_y,B_y,C_y,D_y,E_y,answer
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",722,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",A
1,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",1709,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",A
2,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",156,"Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",B
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,935,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,E
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,965,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277,495,What is the main focus of cryogenic and noble ...,Distinguishing background particles from dark ...,Detecting the heat produced when a particle hi...,Detecting the heat produced when a particle hi...,Detecting the heat produced when a particle hi...,Detecting the mass and interaction cross secti...,1878,Distinguishing background particles from dark ...,Detecting the heat produced when a particle hi...,Detecting the heat produced when a particle hi...,Detecting the heat produced when a particle hi...,Detecting the mass and interaction cross secti...,A
278,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.,145,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as 

> observation:
* the test_df has 282 common rows in like in the train df.
* so we can directly use the answers to partially fill the submission for those rows
* will do it after the model has generated it's predictions

In [6]:
# mapping and dropping 
OPTIONS = ["A", "B", "C", "D", "E"]
label2id = {opt: i for i, opt in enumerate(OPTIONS)}
id2label = {v: k for k, v in label2id.items()}

df['labels'] = df['answer'].apply(lambda x: label2id[x])
df = df.drop(columns=['id','answer'], axis=1)
test_df = test_df.drop(columns=['id'], axis=1)

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
from datasets import Dataset

# test_df['labels'] = test_df['answer'].apply(lambda x: label2id[x])

In [8]:
test_df.columns, df.columns

(Index(['prompt', 'A', 'B', 'C', 'D', 'E'], dtype='object'),
 Index(['prompt', 'A', 'B', 'C', 'D', 'E', 'labels'], dtype='object'))

# Preprocess function

In [9]:
def map_labels(string):
    label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
    return label2id[string]

def preprocess_function(examples):
    prompts = [[context] * 5 for context in examples["prompt"]]
    choices = [
        [examples[choice][i] for choice in ["A", "B", "C", "D", "E"]] 
        for i in range(len(examples["prompt"]))
    ]

    prompts = sum(prompts, [])
    choices = sum(choices, [])
    
    tokenized_examples = tokenizer(prompts, choices, truncation=True, padding="max_length")     
    
    output = {
        k: [v[i : i + 5] for i in range(0, len(v), 5)]
        for k, v in tokenized_examples.items()
    }

    if 'labels' in list(examples.keys()):
        output['labels'] = examples['labels']

    return output 
    
from typing import Any, Optional
from dataclasses import dataclass

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: Any
    padding: bool = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "labels"
        labels = []
        
        if label_name in features[0].keys():
            raw_labels = [feature.pop(label_name) for feature in features]
            letter_to_id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
            for l in raw_labels:
                if isinstance(l, str) and l in letter_to_id:
                    labels.append(letter_to_id[l])
                else:
                    labels.append(int(l))
        else:
            labels = None
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] 
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(

            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
            
        return batch

In [10]:
import numpy as np

def map3_score(labels, logits):
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    if logits.ndim == 1:
        logits = logits.reshape(-1, 5) # reshaping to, [a b c d e]

    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    
    for lab, ranked in zip(labels, top3):
        if lab == ranked[0]:
            scores.append(1.0)
        elif lab == ranked[1]:
            scores.append(0.5)
        elif lab == ranked[2]:
            scores.append(1 / 3)
        else:
            scores.append(0.0)
            
    return float(np.mean(scores))

def compute_metrics(eval_pred) -> dict:
    logits, labels = eval_pred  
    
    return {
        "map3": map3_score(labels, logits)
    }

# wandb init

In [11]:
import wandb 
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")

IS_WANDB = wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: better-than-you (iitm-dlgenai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Model Trainings
---
## Model 1

config

In [12]:
MODEL_NAME = "google-bert/bert-base-uncased"
OPTIONS = ["A", "B", "C", "D", "E"]
NUM_CHOICES = len(OPTIONS)
BATCH_SIZE = 100
EPOCHS = 20

In [13]:
from transformers import AutoModelForMultipleChoice, AutoTokenizer

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
from datasets import Dataset


full_dataset = Dataset.from_pandas(df)
split = full_dataset.train_test_split(test_size=0.2, seed=42)
# print(split)

encoded_train = split['train'].map(
    preprocess_function,
    batched=True,
    remove_columns=[col for col in split['train'].column_names if col != "answer"]
)

encoded_val = split['test'].map(
    preprocess_function,
    batched=True,
    remove_columns=[col for col in split['test'].column_names if col != "answer"]
)


dt = Dataset.from_pandas(test_df)
encoded_test = dt.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dt.column_names,
)

del full_dataset, split, dt

encoded_train.shape, encoded_val.shape, encoded_test.shape

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

((1600, 4), (400, 4), (500, 3))

In [15]:
encoded_train

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1600
})

In [16]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./google-wala-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="map3",
    # learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="wandb",
    logging_steps=50,
    run_name="google-wala-model",
)

wandb.init(project="24f3004148-t22026")

data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)
trainer1 = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)
trainer1.train()
# wandb.finish()


wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260731_182951-ym9e5ekj
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pleasant-universe-16
wandb: ⭐️ View project at https://wandb.ai/iitm-dlgenai-project/24f3004148-t22026
wandb: 🚀 View run at https://wandb.ai/iitm-dlgenai-project/24f3004148-t22026/runs/ym9e5ekj
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map3
1,1.276785,0.817446,0.945000
2,0.343065,0.217906,0.995000
3,0.123195,0.070327,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.9754930734634399, metrics={'train_runtime': 1407.1771, 'train_samples_per_second': 3.411, 'train_steps_per_second': 0.426, 'total_flos': 6314608631808000.0, 'train_loss': 0.9754930734634399, 'epoch': 3.0})

cleanup

In [17]:
import gc
import torch

preds = trainer1.predict(encoded_test)
raw_logits_google = preds.predictions
raw_logits_google = raw_logits_google.copy()

del model, tokenizer
del trainer1
del encoded_train, encoded_val, encoded_test

gc.collect()
torch.cuda.empty_cache()

---
## Model 2
### Config

In [18]:
MODEL_NAME = "roberta-base"
NUM_CHOICES = len(OPTIONS)
BATCH_SIZE = 100
EPOCHS = 20

In [19]:
from transformers import AutoModelForMultipleChoice, AutoTokenizer

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
from datasets import Dataset


full_dataset = Dataset.from_pandas(df)
split = full_dataset.train_test_split(test_size=0.2, seed=42)
# print(split)

encoded_train = split['train'].map(
    preprocess_function,
    batched=True,
    remove_columns=[col for col in split['train'].column_names if col != "answer"]
)

encoded_val = split['test'].map(
    preprocess_function,
    batched=True,
    remove_columns=[col for col in split['test'].column_names if col != "answer"]
)

dt = Dataset.from_pandas(test_df)
encoded_test = dt.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dt.column_names,
)

del full_dataset, split, dt

encoded_train.shape, encoded_test.shape, encoded_val.shape

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

((1600, 3), (500, 2), (400, 3))

In [21]:
training_args = TrainingArguments(
    output_dir="./roberta-wala-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="map3",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="wandb",
    logging_steps=50,
    run_name="roberta-wala-model",
)

# wandb.init(project="24f3004148-t22026")

data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)
trainer2  = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)
trainer2.train()
# wandb.finish()

A ConfigError was raised whilst setting the number of model parameters in Weights & Biases config.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map3
1,1.843410,1.158874,0.912500
2,0.984576,0.648395,0.977500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=400, training_loss=1.8877544212341308, metrics={'train_runtime': 933.956, 'train_samples_per_second': 3.426, 'train_steps_per_second': 0.428, 'total_flos': 4209739087872000.0, 'train_loss': 1.8877544212341308, 'epoch': 2.0})

In [22]:
import gc
import torch

preds = trainer2.predict(encoded_test)
raw_logits_roberta = preds.predictions
raw_logits_roberta = raw_logits_roberta.copy()

del model, tokenizer
del trainer2
del encoded_train, encoded_val, encoded_test

gc.collect()
torch.cuda.empty_cache()

---
## Model 3
### Config

In [23]:
OPTIONS = ["A", "B", "C", "D", "E"]
NUM_CHOICES = len(OPTIONS)
BATCH_SIZE = 100
EPOCHS = 20

In [24]:
# builiding the tokenizer
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
from transformers import PreTrainedTokenizerFast

def batched_generator(dataset, batch_size=1000):
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i : i + batch_size]
        combined_texts = []
        for p, a, b, c, d, e in zip(
            batch["prompt"], batch["A"], batch["B"], batch["C"], batch["D"], batch["E"]
        ):
            text = f"{p} [SEP] {a} [SEP] {b} [SEP] {c} [SEP] {d} [SEP] {e}"
            combined_texts.append(text)
        yield combined_texts

tokenizer_model = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer_model.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer_model.decoder = decoders.ByteLevel()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=5000,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[SEP]"],
)

text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']
combined_df = pd.concat([df[text_columns], test_df[text_columns]], ignore_index=True)

tokenizer_dataset = Dataset.from_pandas(combined_df)
tokenizer_model.train_from_iterator(batched_generator(tokenizer_dataset), trainer=bpe_trainer)

pad_id = tokenizer_model.token_to_id("[PAD]")
tokenizer_model.enable_padding(pad_id=pad_id, pad_token="[PAD]", length=256)
tokenizer_model.enable_truncation(max_length=256)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_model,
    pad_token="[PAD]",
    unk_token="[UNK]",
    bos_token="[BOS]",
    eos_token="[EOS]",
    sep_token="[SEP]",
    model_max_length=256,
    padding_side="right",
    truncation_side="right"
)

In [25]:
from torch import nn

class MLPClassifierHF(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_classes=5, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
                
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            # nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        embs = self.embedding(input_ids)                      
        mask = attention_mask.unsqueeze(-1).float()          
        sum_embs = (embs * mask).sum(dim=1)                 
        avg_embs = sum_embs / mask.sum(dim=1).clamp(min=1)  
        logits = self.net(avg_embs)                          

        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

model = MLPClassifierHF(
    vocab_size=tokenizer.vocab_size,
    embed_dim=32,
    num_classes=5,
    hidden_dim=64,
    pad_idx=tokenizer.pad_token_id,
)

In [26]:
def preprocess_dataset(batch):
    texts = [
        f"Question: {p} Choice A: {a} Choice B: {b} Choice C: {c} Choice D: {d} Choice E: {e}"
        for p, a, b, c, d, e in zip(
            batch['prompt'], batch['A'], batch['B'], batch['C'], batch['D'], batch['E']
        )
    ]
    # print(texts[0])
    output = tokenizer(
        texts, 
        padding=True,
        truncation=True, 
        max_length=256
    )
    
    if 'labels' in batch:
        output['labels'] = batch['labels']
        
    return output

dataset = Dataset.from_pandas(df)
encoded_dataset = dataset.map(
    preprocess_dataset,
    batched=True,
    batch_size=100,
    remove_columns=dataset.column_names
)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [27]:
# MANUAL FIXING OF TEST_ENCODE INPUT IDS SHAPE MISMATCH

flat_input_ids = []
flat_attention_masks = []
for _, row in test_df.iterrows():
    # one string per q in test
    text = f"Question: {row['prompt']} Choice A: {row['A']} Choice B: {row['B']} Choice C: {row['C']} Choice D: {row['D']} Choice E: {row['E']}"
    
    encoded = tokenizer(
        text, 
        truncation=True, 
        max_length=256, 
        padding="max_length"
    )
    
    flat_input_ids.append(encoded['input_ids'])
    flat_attention_masks.append(encoded['attention_mask'])

encoded_test = Dataset.from_dict({
    "input_ids": torch.tensor(flat_input_ids, dtype=torch.long),
    "attention_mask": torch.tensor(flat_attention_masks, dtype=torch.long)
})
encoded_test.set_format("torch")
encoded_test, encoded_test['input_ids'][0].shape

(Dataset({
     features: ['input_ids', 'attention_mask'],
     num_rows: 500
 }),
 torch.Size([256]))

In [28]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./custom_wala_model",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    num_train_epochs=100,
    weight_decay=0.01,
    report_to="wandb",
    logging_steps=50,
    run_name="custom-wala-model",
)

trainer3 = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset,
    processing_class=tokenizer,
)

trainer3.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,1.602592
100,1.409001
150,1.289689
200,1.136269
250,0.983228
300,0.807096
350,0.688276
400,0.524302
450,0.399323
500,0.305747


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=12500, training_loss=0.04218554774198681, metrics={'train_runtime': 181.4226, 'train_samples_per_second': 1102.399, 'train_steps_per_second': 68.9, 'total_flos': 0.0, 'train_loss': 0.04218554774198681, 'epoch': 100.0})

In [29]:
import gc
import torch

preds = trainer3.predict(encoded_test)
raw_logits_custom = preds.predictions
raw_logits_custom = raw_logits_custom.copy()

del model, tokenizer
del trainer3

gc.collect()
torch.cuda.empty_cache()

# Test predictions & Submission

In [30]:
# at this point i should have all the 3 raw_logits
# so i will directly use the raw_logits here
# for now im importing them

try:
    print(len(raw_logits_custom))
    print(len(raw_logits_google))
    print(len(raw_logits_roberta))
except Exception as e:
    print(e)
    
# works


500
500
500


In [31]:
# now creating a ensemble for all the models
w1, w2, w3 = 0.4, 0.3, 0.3
# custom, google, roberta

prob1 = torch.softmax(torch.tensor(raw_logits_custom), dim=1)
prob2 = torch.softmax(torch.tensor(raw_logits_google), dim=1)
prob3 = torch.softmax(torch.tensor(raw_logits_roberta), dim=1)

ensemble_probs = (w1 * prob1) + (w2 * prob2) + (w3 * prob3)

top3_indices = torch.topk(ensemble_probs, k=3, dim=1).indices.tolist()

label_map = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

predictions = [
    f"{label_map[row[0]]} {label_map[row[1]]} {label_map[row[2]]}" 
    for row in top3_indices
]

submission_df = pd.DataFrame({
    "ID": range(1, 1 + len(test_df)), 
    "Prediction": predictions
})

In [32]:
# and finally replace the rows with duplicate rows of train in the submission

def format_exact_match(answer):
    all_labels = ['A', 'B', 'C', 'D', 'E']
    
    # remaining answers
    remaining = [lbl for lbl in all_labels if lbl != answer]
    return f"{answer} {remaining[0]} {remaining[1]}"

# mapping dict
override_dict = {
    row['id_x']: format_exact_match(row['answer']) 
    for _, row in similar_rows.iterrows()
}

# onlu update the matching ids
mask = submission_df['ID'].isin(override_dict.keys())
submission_df.loc[mask, 'Prediction'] = submission_df.loc[mask, 'ID'].map(override_dict)

In [33]:
submission_df.to_csv("submission.csv", index=False)

In [34]:
import shutil
import os

# 1. Verify submission file exists
if os.path.exists('submission.csv'):
    print("submission.csv found. Cleaning up temporary files...")

    folders_to_delete = [
        './custom_wala_model',
        './google_wala_model',
        './roberta_wala_model'
    ]

    for folder in folders_to_delete:
        if os.path.exists(folder):
            shutil.rmtree(folder)
            print(f"Deleted {folder}")

    print("Cleanup complete.")
else:
    print("Error: submission.csv was not generated. Skipping cleanup.")

submission.csv found. Cleaning up temporary files...
Deleted ./custom_wala_model
Cleanup complete.
